# EDM U-Net: capacity sweep across training seeds

**Prof. Baptista's ask 3, verbatim:** *"hold off on Q3 conclusions until the rerun with
multiple seeds."*

This is `edm_unet_capacity_sweep.ipynb` with **exactly one thing changed**: the training seed
passed to `train_edm`. Architecture set, `n_train`, step budget, checkpoint schedule, batch
size, sampler, latents, metric and reference draws are all byte-identical to that notebook.
Nothing in `src/` is modified.

It answers two questions at once:

1. **The stated ask.** Does the capacity ordering reported in the single-seed sweep survive
   re-running? That sweep found the gap below the neutral line ordered monotonically in
   parameter count at 30k steps, with the two smallest networks ending *above* the line.
2. **The prerequisite nobody has measured.** Every null in this project is currently declared
   against an assumed single-seed spread of 0.03 to 0.06 on the coarse band. That number has
   never been measured: no experiment in the repo varies the training seed. The per-config
   standard deviation across seeds produced here *is* that number, and it is what makes the
   transition, capacity and dimension-probe results interpretable rather than merely stated.

**Read the seed spread first, the ordering second.** If the spread turns out to be comparable
to the 0.059 full spread across configurations, the ordering is not resolved and should be
reported as such rather than as a capacity trend.

**Device note.** `resolve_device()` picks CUDA on a cluster and MPS on the laptop, and the two
do not agree bit-for-bit. Every seed here, including seed 0, must be run on the *same* device,
and these numbers must not be compared against the committed single-seed run from a different
device. Doing so would confound the seed effect with a backend change.


In [1]:
import sys, os, math, time, copy
import numpy as np
import torch
import matplotlib.pyplot as plt

# -- path setup --
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import diffusion_score_models as score_models
from multiband_data_utils import generate_multiband_dataset_postmask
from memorization_metrics import RingMetricContext
from edm import EDMPrecond, EDMScoreWrapper, train_edm
from unet import SmallUNet, count_parameters
from device_utils import resolve_device

DEVICE = resolve_device()   # cuda > mps > cpu; use resolve_device("cpu") on the personal laptop
print(f'device: {DEVICE}')

device: cuda


In [2]:
# -- Data generation (identical config to the earlier memorization notebooks) --
components = [
    {"name": "coarse", "length_scale": 2.0,  "s": 2.0, "sigma_sq": 1.0, "band": (0.5, 4.0)},
    {"name": "mid1",   "length_scale": 6.0,  "s": 2.0, "sigma_sq": 1.0, "band": (4.0, 10.0)},
    {"name": "mid2",   "length_scale": 12.0, "s": 2.0, "sigma_sq": 1.0, "band": (10.0, 18.0)},
    {"name": "fine",   "length_scale": 24.0, "s": 2.0, "sigma_sq": 1.0, "band": (18.0, 32.0)},
]
result = generate_multiband_dataset_postmask(
    num_samples=200, grid_size=128, components=components,
    weights=[1.0, 0.8, 0.8, 1.2], seed=42, normalize=True,
)
bands = result.get('bands', {c['name']: c['band'] for c in components})
N = 128
x_all = result['combined']          # kept on CPU; slices moved to DEVICE as needed
ctx = RingMetricContext(N, bands, device=DEVICE)
print(f'x_all: {tuple(x_all.shape)}')

x_all: (200, 128, 128)


In [3]:
# -- Evaluation: matched sampler (sigma_max=10, config D of the sigma-fix study) --
VE_SAMPLE = score_models.VE_EDM(sigma_min=0.002, sigma_max=10.0)
N_GEN = 16
N_SDE_STEPS = 1000
N_RAND_REF = 32
LATENT_SEED = 42

@torch.no_grad()
def sample_from(score_fn):
    torch.manual_seed(LATENT_SEED)   # same latents (and step noise stream) for every run
    latents = torch.randn(N_GEN, N*N, device=DEVICE)
    out = VE_SAMPLE.SDEsampler(score_fn, latents, num_steps=N_SDE_STEPS)
    return out.reshape(N_GEN, N, N)

@torch.no_grad()
def pixel_nn_stats(x_gen, x_train):
    '''Relative pixel-space L2 distance to the nearest training field (Baptista-style
    collapse measure). Returns per-sample distances; threshold at plot time.'''
    d = torch.cdist(x_gen.flatten(1), x_train.flatten(1))
    nn_rel = d.min(dim=1).values / x_train.flatten(1).norm(dim=1).mean()
    return nn_rel.cpu()

@torch.no_grad()
def evaluate_checkpoint(precond, x_train):
    wrapper = EDMScoreWrapper(precond, VE_SAMPLE.marginal_prob_std, N, c_tikhonov=0.0).to(DEVICE)
    x_gen = sample_from(wrapper)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {
        'coarse_score': m['coarse_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
        'mean_ratio': m['mean_ratio'].cpu(),
        'nn_rel': pixel_nn_stats(x_gen, x_train),
        'samples': x_gen[:2].cpu(),
    }

@torch.no_grad()
def gmm_reference(x_train):
    train_flat = x_train.reshape(x_train.shape[0], -1)
    gmm = score_models.GMM_score(train_flat, VE_SAMPLE.marginal_prob_mean,
                                 VE_SAMPLE.marginal_prob_std)
    x_gen = sample_from(gmm)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {
        'coarse_score': m['coarse_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
        'nn_rel': pixel_nn_stats(x_gen, x_train),
    }

results_dir = os.path.join(repo_root, 'results', 'data')
os.makedirs(results_dir, exist_ok=True)
fig_dir = os.path.join(repo_root, 'results', 'figures')
os.makedirs(fig_dir, exist_ok=True)

## Sweep configuration

`SEEDS` is the only new knob. Three seeds give a usable spread estimate at 3x the cost of the
original sweep (18 training runs against 6). Raising it to 5 is a one-line change and is worth
doing if the queue allows, since the standard deviation of 3 samples is itself noisy.


In [4]:
# -- Sweep configuration: identical to edm_unet_capacity_sweep.ipynb except SEEDS --
SMOKE = False    # True: tiny end-to-end run to verify the pipeline

SEEDS = (0, 1, 2)        # <-- the only change from the single-seed sweep

N_TRAIN = 8
TOTAL_STEPS = 30000
CHECKPOINT_AT = [250, 500, 1000, 2000, 4000, 8000, 16000, 30000]
BATCH_SIZE = 8

CONFIGS = [
    dict(base_channels=8,  emb_dim=64, num_levels=3),
    dict(base_channels=16, emb_dim=64, num_levels=3),   # baseline (= transition notebook)
    dict(base_channels=32, emb_dim=64, num_levels=3),
    dict(base_channels=64, emb_dim=64, num_levels=3),
    dict(base_channels=16, emb_dim=64, num_levels=2),
    dict(base_channels=16, emb_dim=64, num_levels=4),
]

if SMOKE:
    TOTAL_STEPS = 200
    CHECKPOINT_AT = [100, 200]
    N_SDE_STEPS = 50
    N_GEN = 4
    CONFIGS = CONFIGS[:2]
    SEEDS = (0, 1)

def cfg_name(cfg):
    return f"C{cfg['base_channels']}_L{cfg['num_levels']}"

for cfg in CONFIGS:
    cfg['n_params'] = count_parameters(SmallUNet(**cfg))
    print(f"{cfg_name(cfg):>8s}: {cfg['n_params']:>10,} params")

x_train = x_all[:N_TRAIN].to(DEVICE)
train_flat = x_train.reshape(N_TRAIN, -1)
ckpt_path = os.path.join(results_dir, 'capacity_multiseed_checkpoints.pt')
print(f"\n{len(CONFIGS)} configs x {len(SEEDS)} seeds = {len(CONFIGS)*len(SEEDS)} training runs")
print(f"device: {DEVICE}  (all seeds must run on this same device)")


   C8_L3:     64,345 params
  C16_L3:    195,697 params
  C32_L3:    709,153 params
  C64_L3:  2,739,073 params
  C16_L2:     58,641 params
  C16_L4:    729,905 params

6 configs x 3 seeds = 18 training runs
device: cuda  (all seeds must run on this same device)


In [5]:
# -- Training: (config, seed) pairs, checkpointed incrementally so the job is restartable --
all_ckpts = {}
if os.path.exists(ckpt_path):
    all_ckpts = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'resuming: {len(all_ckpts)} (config, seed) pairs already trained')

for seed in SEEDS:
    for cfg in CONFIGS:
        key = (cfg_name(cfg), seed)
        if key in all_ckpts:
            print(f'{key}: already trained, skipping')
            continue
        print(f"===== training {cfg_name(cfg)} seed {seed} ({cfg['n_params']:,} params) =====")
        make_unet = lambda base_channels, emb_dim: SmallUNet(
            base_channels=base_channels, emb_dim=emb_dim, num_levels=cfg['num_levels'])
        t0 = time.time()
        saved = train_edm(train_flat, grid_size=N, total_steps=TOTAL_STEPS,
                          checkpoint_at=CHECKPOINT_AT, base_channels=cfg['base_channels'],
                          emb_dim=cfg['emb_dim'], lr=1e-3, batch_size=BATCH_SIZE,
                          seed=seed, device=DEVICE, UNetClass=make_unet)
        all_ckpts[key] = {
            'config': {k: cfg[k] for k in ('base_channels', 'emb_dim', 'num_levels', 'n_params')},
            'seed': seed,
            'ckpts': {step: {'state_dict': {k: v.cpu() for k, v in p.state_dict().items()},
                             'sigma_data': p.sigma_data}
                      for step, p in saved.items()},
        }
        torch.save(all_ckpts, ckpt_path)
        print(f'  {time.time()-t0:.0f}s; saved -> {ckpt_path}')


===== training C8_L3 seed 0 (64,345 params) =====
Estimated sigma_data = 0.9810


  step 1/30000  loss=1.581663


  >> checkpoint saved at step 250


  step 500/30000  loss=0.219234
  >> checkpoint saved at step 500


  step 1000/30000  loss=0.130366
  >> checkpoint saved at step 1000


...

  step 28500/30000  loss=0.007763


  step 29000/30000  loss=0.012434


  step 29500/30000  loss=0.008545


  step 30000/30000  loss=0.002312
  >> checkpoint saved at step 30000


  1012s; saved -> /project/6109267/ishiyer/research-diffusion/results/data/capacity_multiseed_checkpoints.pt


In [6]:
# -- Evaluation: identical metric, sampler and latents for every (config, seed) --
gmm_ref = gmm_reference(x_train)
print(f"GMM ceiling (n_train={N_TRAIN}): coarse={gmm_ref['coarse_score']:.5f} "
      f"fine={gmm_ref['fine_score']:.4f} collapse={(gmm_ref['nn_rel']<0.3).float().mean():.2f}")

# Neutral line: held-out real fields through the identical metric. Same slice as every other
# notebook (x_all[100:100+N_GEN]), so the gap column is comparable across the project.
@torch.no_grad()
def neutral_reference(x_train, n_seeds=5):
    hold = x_all[100:100+N_GEN].to(DEVICE)
    cs, fs = [], []
    for s in range(n_seeds):
        torch.manual_seed(s)
        m = ctx.evaluate(hold, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
        cs.append(m['coarse_score'].mean().item()); fs.append(m['fine_score'].mean().item())
    return {'coarse_score': float(np.mean(cs)), 'coarse_std': float(np.std(cs)),
            'fine_score': float(np.mean(fs)), 'fine_std': float(np.std(fs))}

neutral = neutral_reference(x_train)
print(f"neutral line (5 reference-draw seeds): coarse={neutral['coarse_score']:.4f} "
      f"+/- {neutral['coarse_std']:.4f}")

eval_results = {}
for key, entry in all_ckpts.items():
    cfg = entry['config']; name, seed = key
    print(f"===== evaluating {name} seed {seed} =====")
    eval_results[key] = {}
    for step, ck in sorted(entry['ckpts'].items()):
        unet = SmallUNet(base_channels=cfg['base_channels'], emb_dim=cfg['emb_dim'],
                         num_levels=cfg['num_levels']).to(DEVICE)
        precond = EDMPrecond(unet, sigma_data=ck['sigma_data']).to(DEVICE)
        precond.load_state_dict(ck['state_dict']); precond.eval()
        r = evaluate_checkpoint(precond, x_train)
        eval_results[key][step] = r
        print(f"  step {step:>6}: coarse={r['coarse_score']:.4f} "
              f"gap={r['coarse_score']-neutral['coarse_score']:+.4f} "
              f"collapse={(r['nn_rel']<0.3).float().mean():.2f}")

torch.save({'eval_results': eval_results, 'gmm_ref': gmm_ref, 'neutral': neutral,
            'configs': {cfg_name(c): c for c in CONFIGS}, 'seeds': list(SEEDS),
            'n_train': N_TRAIN, 'checkpoint_at': CHECKPOINT_AT,
            'sampler': {'sigma_max': 10.0, 'n_steps': N_SDE_STEPS, 'latent_seed': LATENT_SEED},
            'device': str(DEVICE),
            'note': ('Capacity sweep across training seeds. Identical to '
                     'edm_unet_capacity_sweep.ipynb except train_edm(seed=...). Reports the '
                     'per-config seed spread, which is the significance threshold the rest of '
                     'the project has been assuming without measuring.')},
           os.path.join(results_dir, 'capacity_multiseed.pt'))
print('saved -> capacity_multiseed.pt')


GMM ceiling (n_train=8): coarse=0.00008 fine=0.0085 collapse=1.00
neutral line (5 reference-draw seeds): coarse=0.8477 +/- 0.0064
===== evaluating C8_L3 seed 0 =====


  step    250: coarse=0.8445 gap=-0.0032 collapse=0.00


  step    500: coarse=0.8293 gap=-0.0184 collapse=0.00


  step   1000: coarse=0.8352 gap=-0.0125 collapse=0.00


  step   2000: coarse=0.8360 gap=-0.0117 collapse=0.00


...

  step   2000: coarse=0.8360 gap=-0.0117 collapse=0.00


  step   4000: coarse=0.8179 gap=-0.0298 collapse=0.00


  step   8000: coarse=0.8130 gap=-0.0347 collapse=0.00


  step  16000: coarse=0.8086 gap=-0.0391 collapse=0.00


  step  30000: coarse=0.8003 gap=-0.0474 collapse=0.00
saved -> capacity_multiseed.pt


In [7]:
# -- The headline number: per-config spread across seeds at the final checkpoint --
names = [cfg_name(c) for c in CONFIGS]
params = {cfg_name(c): c['n_params'] for c in CONFIGS}
nb_ = neutral['coarse_score']
FINAL = max(CHECKPOINT_AT)   # 30000 in a real run

print(f"neutral coarse line {nb_:.4f}   |   gap = score - neutral, negative = toward copying\n")
print(f"{'config':>8} {'params':>10} " + " ".join(f"{'s'+str(s):>8}" for s in SEEDS)
      + f" {'mean':>8} {'std':>8} {'range':>8}")
rows = {}
for name in names:
    gs = [eval_results[(name, s)][FINAL]['coarse_score'] - nb_ for s in SEEDS
          if (name, s) in eval_results]
    if not gs: continue
    rows[name] = gs
    print(f"{name:>8} {params[name]:>10,} " + " ".join(f"{g:>+8.4f}" for g in gs)
          + f" {np.mean(gs):>+8.4f} {np.std(gs):>8.4f} {max(gs)-min(gs):>8.4f}")

pooled = float(np.mean([np.std(g) for g in rows.values()]))
worst  = float(max(max(g)-min(g) for g in rows.values()))
across = max(np.mean(g) for g in rows.values()) - min(np.mean(g) for g in rows.values())

print(f"\nSEED SPREAD (the number this experiment exists to produce)")
print(f"  pooled within-config std across seeds : {pooled:.4f}")
print(f"  worst within-config range             : {worst:.4f}")
print(f"  spread of seed-means across configs   : {across:.4f}")
print(f"\n  Assumed throughout the project so far : 0.03 to 0.06 (never measured)")
if worst >= across:
    print("  => Seed noise is at least as large as the capacity effect. The ordering in the")
    print("     single-seed sweep is NOT resolved. Report it as unresolved.")
else:
    print("  => The capacity effect exceeds seed noise. The ordering is resolved at these seeds.")

order_by_params = sorted(rows, key=lambda n: params[n])
seed_mono = {s: all(eval_results[(order_by_params[i], s)][FINAL]['coarse_score']
                    > eval_results[(order_by_params[i+1], s)][FINAL]['coarse_score']
                    for i in range(len(order_by_params)-1)) for s in SEEDS
             if all((n, s) in eval_results for n in order_by_params)}
print(f"\n  Monotone in parameter count at the final checkpoint, per seed: {seed_mono}")
print(f"  ({sum(seed_mono.values())} of {len(seed_mono)} seeds reproduce the reported ordering)")

collapse = max((eval_results[k][s]['nn_rel'] < 0.3).float().mean().item()
               for k in eval_results for s in eval_results[k])
print(f"\n  Max pixel collapse fraction over every cell: {collapse:.4f} "
      f"(GMM through the same pipeline: {(gmm_ref['nn_rel']<0.3).float().mean():.2f})")


neutral coarse line 0.8477   |   gap = score - neutral, negative = toward copying

  config     params       s0       s1       s2     mean      std    range
   C8_L3     64,345  -0.0056  -0.0084  -0.0126  -0.0089   0.0028   0.0069
  C16_L3    195,697  -0.0107  -0.0204  -0.0347  -0.0219   0.0099   0.0240
  C32_L3    709,153  -0.0134  +0.0053  -0.0069  -0.0050   0.0077   0.0187
  C64_L3  2,739,073  -0.0070  -0.0171  -0.0091  -0.0111   0.0044   0.0102
  C16_L2     58,641  -0.0028  -0.0002  -0.0182  -0.0071   0.0080   0.0180
  C16_L4    729,905  -0.0892  -0.0375  -0.0474  -0.0580   0.0224   0.0517

SEED SPREAD (the number this experiment exists to produce)
  pooled within-config std across seeds : 0.0092
  worst within-config range             : 0.0517
  spread of seed-means across configs   : 0.0530

  Assumed throughout the project so far : 0.03 to 0.06 (never measured)
  => The capacity effect exceeds seed noise. The ordering is resolved at these seeds.

  Monotone in parameter count at

In [8]:
# -- Plot: seed band per config --
import matplotlib.cm as cm
lo, hi = math.log10(min(params.values())), math.log10(max(params.values()))
color_of = lambda n: cm.viridis((math.log10(params[n]) - lo) / max(hi - lo, 1e-9))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
for name in names:
    if name not in rows: continue
    steps = sorted(eval_results[(name, SEEDS[0])].keys())
    curves = np.array([[eval_results[(name, s)][st]['coarse_score'] for st in steps]
                       for s in SEEDS if (name, s) in eval_results])
    ax.fill_between(steps, curves.min(0), curves.max(0), color=color_of(name), alpha=0.18)
    ax.plot(steps, curves.mean(0), marker='o', ms=3.5, color=color_of(name),
            ls='--' if name.endswith(('L2', 'L4')) else '-',
            label=f'{name} ({params[name]/1e3:.0f}k)')
ax.axhline(nb_, color='black', lw=1.0, ls=':')
ax.set_xscale('log'); ax.set_xlabel('training step')
ax.set_ylabel('coarse band score')
ax.set_title('shaded = min/max across seeds; dotted = neutral line')
ax.legend(fontsize=7)

ax = axes[1]
xs = [params[n] for n in names if n in rows]
ms = [np.mean(rows[n]) for n in names if n in rows]
es = [np.std(rows[n]) for n in names if n in rows]
ax.errorbar(xs, ms, yerr=es, fmt='o', capsize=4, color='tab:blue')
ax.axhline(0.0, color='black', lw=1.0, ls=':')
ax.axhspan(-pooled, pooled, color='gray', alpha=0.15, label=f'+/- measured seed std ({pooled:.3f})')
ax.set_xscale('log'); ax.set_xlabel('parameters')
ax.set_ylabel('gap from neutral at the final checkpoint')
ax.set_title('capacity effect against measured seed noise')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'capacity_multiseed.png'), dpi=150, bbox_inches='tight')
plt.show()


## Reading this

The decision rule is stated before the numbers are seen, so that it cannot be fitted to them.

- **If the worst within-config range across seeds is at least as large as the spread of
  seed-means across configs**, the capacity ordering reported from the single-seed sweep is
  inside seed noise. The correct write-up is that the ordering is unresolved, not that
  capacity has no effect and not that it has one.
- **If it is clearly smaller**, the ordering is real at these seeds and the single-seed result
  stands.

Either outcome supplies the pooled within-config standard deviation, which is the significance
threshold that sections 5, 6 and 8 of `notes/results_table.md` currently assume rather than
measure. That number should replace the assumed 0.03 to 0.06 everywhere it appears.

The pixel collapse fraction is reported alongside and needs no threshold at all: it is 0.00
across every cell of the single-seed sweep while the closed-form score through the identical
pipeline gives 1.00. If that survives here across seeds, the central negative result does not
depend on the spread measurement in any way.
